<a href="https://colab.research.google.com/github/rathodgayathri9-glitch/neural-networks-machine-learning-exercises/blob/main/Exercise-04-ADALINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
Exercise 4: ADALINE-Based Adaptive Pattern Classification and Prediction System (Iris dataset)

Classes  : Iris Setosa = 0, Iris Versicolor = 1  (Iris Virginica is discarded)
Features : Sepal Length, Sepal Width, Petal Length, Petal Width
Models   : ADALINE, Single-Layer ADALINE (several linear outputs) and Multi-Input ADALINE,
           all trained with the Widrow-Hoff / LMS (delta) rule on the *linear* output.
           Classification threshold on the linear output = 0.5 (targets are 0 / 1).

Usage    : python exercise4_adaline_iris.py [path_to_iris_csv]
           Looks for Iris.csv / iris.csv / iris.data in the current folder. If none is found,
           scikit-learn's built-in copy of the Iris dataset is used (if scikit-learn is installed).
Needs    : numpy, pandas  (scikit-learn only as an optional fallback for loading data)
"""
import os
import sys
import numpy as np
import pandas as pd

FEATURES = ["Sepal Length", "Sepal Width", "Petal Length", "Petal Width"]
CLASS_NAME = {0: "Iris-setosa", 1: "Iris-versicolor"}


# ----------------------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------------------
def load_iris_binary(path=None):
    cands = ([path] if path else []) + ["Iris.csv", "iris.csv", "iris.data", "iris.data.csv"]
    f = next((p for p in cands if p and os.path.exists(p)), None)
    if f is not None:
        df = pd.read_csv(f, header=None if f.endswith(".data") else 0).dropna(how="all")
        df = df[[c for c in df.columns if str(c).strip().lower() not in ("id", "unnamed: 0")]]
        species = df.iloc[:, -1].astype(str).str.lower()
        keep = species.str.contains("setosa") | species.str.contains("versicolor")
        X = df.iloc[:, :4].apply(pd.to_numeric).values[keep.values]
        y = species[keep].str.contains("versicolor").astype(int).values
        return X.astype(float), y, f
    try:
        from sklearn.datasets import load_iris
        d = load_iris()
        keep = d.target < 2
        return d.data[keep].astype(float), d.target[keep].astype(int), "scikit-learn built-in Iris"
    except ImportError:
        raise SystemExit("Iris data not found. Download Iris.csv from Kaggle/UCI and put it next to this "
                         "script, or run:  python exercise4_adaline_iris.py <csv_path>")


def stratified_split(y, test_frac=0.3, seed=42):
    rng = np.random.default_rng(seed)
    te = []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        te.extend(idx[: int(round(len(idx) * test_frac))])
    te = np.array(sorted(te))
    return np.setdiff1d(np.arange(len(y)), te), te


# ----------------------------------------------------------------------------------------
# ADALINE (single or multiple linear output units = single-layer ADALINE)
# ----------------------------------------------------------------------------------------
class Adaline:
    def __init__(self, n_inputs, n_outputs=1, lr=0.01, epochs=100, seed=0, tol=1e-7):
        rng = np.random.default_rng(seed)
        self.W = rng.uniform(-0.05, 0.05, size=(n_inputs, n_outputs))
        self.b = np.zeros(n_outputs)
        self.lr, self.epochs, self.seed, self.tol = lr, epochs, seed, tol
        self.mse_history = []

    def net(self, X):
        return np.atleast_2d(X) @ self.W + self.b            # linear (identity) activation

    def fit(self, X, Y):
        Y = np.asarray(Y, float).reshape(len(X), -1)
        rng = np.random.default_rng(self.seed + 1)
        self.mse_history = []
        for _ in range(self.epochs):
            for i in rng.permutation(len(X)):                 # Widrow-Hoff per-sample update
                err = Y[i] - (X[i] @ self.W + self.b)
                self.W += self.lr * np.outer(X[i], err)
                self.b += self.lr * err
            mse = float(np.mean((Y - self.net(X)) ** 2))
            self.mse_history.append(mse)
            if len(self.mse_history) > 1 and abs(self.mse_history[-2] - mse) < self.tol:
                break
        return self

    def predict_class(self, X):
        return (self.net(X)[:, 0] >= 0.5).astype(int)


# ----------------------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------------------
class App:
    pass


app = App()


def setup(path=None):
    X, y, src = load_iris_binary(path)
    tr, te = stratified_split(y)
    app.X, app.y, app.tr, app.te = X, y, tr, te
    app.mu, app.sd = X[tr].mean(0), X[tr].std(0)
    app.sd[app.sd == 0] = 1
    app.Z = (X - app.mu) / app.sd                            # z-score using training statistics
    print(f"Loaded {src}: {len(y)} samples (50 Setosa + 50 Versicolor), 4 features. "
          f"Train {len(tr)}, test {len(te)}.")


def acc(y, p):
    return float((y == p).mean())


def confusion(y, p):
    return (int(((y == 0) & (p == 0)).sum()), int(((y == 0) & (p == 1)).sum()),
            int(((y == 1) & (p == 0)).sum()), int(((y == 1) & (p == 1)).sum()))


def print_conf(y, p):
    tn, fp, fn, tp = confusion(y, p)
    print("Confusion matrix:        Pred Setosa  Pred Versicolor")
    print(f"   Actual Setosa      {tn:>12} {fp:>16}")
    print(f"   Actual Versicolor  {fn:>12} {tp:>16}")


def ask(prompt, default, cast=float):
    s = input(f"{prompt} [default {default}]: ").strip()
    if not s:
        return default
    try:
        return cast(s)
    except ValueError:
        print("   invalid input, using default.")
        return default


def train_clf(cols=(0, 1, 2, 3), lr=0.01, epochs=100, Z=None, y=None, seed=0):
    Z = app.Z[app.tr] if Z is None else Z
    y = app.y[app.tr] if y is None else y
    return Adaline(len(cols), 1, lr, epochs, seed).fit(Z[:, list(cols)], y)


def noisy(Z, sigma, rng):
    return Z + rng.normal(0, sigma, Z.shape)


def get_main():
    if not hasattr(app, "main"):
        app.main = train_clf()
    return app.main


# ----------------------------------------------------------------------------------------
# 1. Binary Pattern Classification
# ----------------------------------------------------------------------------------------
def opt1():
    print("\n=== 1. Binary Pattern Classification (Setosa = 0, Versicolor = 1) ===")
    lr = ask("Learning rate", 0.01)
    ep = ask("Max epochs", 100, int)
    m = train_clf(lr=lr, epochs=ep)
    app.main = m
    Ztr, Zte = app.Z[app.tr], app.Z[app.te]
    print(f"\nEpochs run: {len(m.mse_history)}")
    for e in sorted({1, 5, 10, 25, 50, len(m.mse_history)}):
        if e <= len(m.mse_history):
            print(f"   epoch {e:>3}: MSE = {m.mse_history[e - 1]:.5f}")
    print("\nLearned weights (standardised inputs):")
    for n, w in zip(FEATURES, m.W[:, 0]):
        print(f"   {n:<14}{w:+.4f}")
    print(f"   {'Bias':<14}{m.b[0]:+.4f}")
    ptr, pte = m.predict_class(Ztr), m.predict_class(Zte)
    print(f"\nTraining accuracy: {acc(app.y[app.tr], ptr) * 100:.2f}%   "
          f"Test accuracy: {acc(app.y[app.te], pte) * 100:.2f}%")
    print_conf(app.y[app.te], pte)
    print("\nFirst 8 test samples:")
    print(f"{'idx':>4} {'actual':>16} {'ADALINE output':>15} {'predicted':>16}")
    out = m.net(Zte)[:, 0]
    for i in range(min(8, len(pte))):
        print(f"{i:>4} {CLASS_NAME[app.y[app.te][i]]:>16} {out[i]:>15.4f} {CLASS_NAME[pte[i]]:>16}")
    s = input("\nPredict a new flower? Enter 4 values (SL,SW,PL,PW in cm) or press Enter to skip: ").strip()
    if s:
        try:
            v = np.array([float(t) for t in s.replace(" ", ",").split(",") if t])
            assert len(v) == 4
            o = m.net((v - app.mu) / app.sd)[0, 0]
            print(f"   ADALINE output = {o:.4f} -> {CLASS_NAME[int(o >= 0.5)]}")
        except Exception:
            print("   could not parse four numbers.")


# ----------------------------------------------------------------------------------------
# 2. Multi-Input ADALINE Feature Prediction (regression, single-layer when >1 target)
# ----------------------------------------------------------------------------------------
def opt2():
    print("\n=== 2. Multi-Input ADALINE Feature Prediction ===")
    print("Predict one or more features from the remaining ones (a single-layer ADALINE has one linear output per target).")
    for i, n in enumerate(FEATURES, 1):
        print(f"   {i}. {n}")
    s = input("Target feature number(s), comma separated [default 4]: ").strip() or "4"
    try:
        tg = sorted({int(t) - 1 for t in s.split(",") if t.strip()})
        assert tg and all(0 <= t < 4 for t in tg) and len(tg) < 4
    except Exception:
        print("   invalid selection, using feature 4.")
        tg = [3]
    src = [j for j in range(4) if j not in tg]
    tr, te = app.tr, app.te
    mdl = Adaline(len(src), len(tg), lr=0.01, epochs=300).fit(app.Z[tr][:, src], app.Z[tr][:, tg])
    pred = mdl.net(app.Z[te][:, src])
    print(f"\nInputs : {[FEATURES[j] for j in src]}")
    print(f"Targets: {[FEATURES[j] for j in tg]}   (epochs run: {len(mdl.mse_history)})")
    for k, t in enumerate(tg):
        actual = app.Z[te][:, t] * app.sd[t] + app.mu[t]
        p = pred[:, k] * app.sd[t] + app.mu[t]
        rmse = float(np.sqrt(np.mean((actual - p) ** 2)))
        r2 = 1 - np.sum((actual - p) ** 2) / np.sum((actual - actual.mean()) ** 2)
        print(f"\n--- {FEATURES[t]} ---  RMSE = {rmse:.3f} cm, R^2 = {r2:.3f}")
        print("   weights: " + ", ".join(f"{FEATURES[j]}={mdl.W[a, k]:+.3f}" for a, j in enumerate(src)))
        print(f"   {'idx':>4} {'actual':>8} {'predicted':>10} {'error':>8}")
        for i in range(min(6, len(actual))):
            print(f"   {i:>4} {actual[i]:>8.2f} {p[i]:>10.2f} {p[i] - actual[i]:>+8.2f}")


# ----------------------------------------------------------------------------------------
# 3. Feature Influence Analyzer
# ----------------------------------------------------------------------------------------
def opt3():
    print("\n=== 3. Feature Influence Analyzer ===")
    m = get_main()
    ytr, yte = app.y[app.tr], app.y[app.te]
    w = np.abs(m.W[:, 0])
    base_mse = float(np.mean((yte - m.net(app.Z[app.te])[:, 0]) ** 2))
    print(f"{'feature':<14}{'weight':>9}{'|w| share':>11}{'only-this-feature':>19}{'drop-this-feature':>19}")
    print(f"{'':<14}{'':>9}{'':>11}{'acc / test MSE':>19}{'acc / test MSE':>19}")
    rows = []
    for j, n in enumerate(FEATURES):
        one = train_clf(cols=(j,), epochs=200)
        p1 = one.predict_class(app.Z[app.te][:, [j]])
        mse1 = np.mean((yte - one.net(app.Z[app.te][:, [j]])[:, 0]) ** 2)
        rest = [c for c in range(4) if c != j]
        dr = train_clf(cols=rest, epochs=200)
        p2 = dr.predict_class(app.Z[app.te][:, rest])
        mse2 = np.mean((yte - dr.net(app.Z[app.te][:, rest])[:, 0]) ** 2)
        rows.append((n, m.W[j, 0], w[j] / w.sum() * 100, acc(yte, p1) * 100, mse1, acc(yte, p2) * 100, mse2))
    for n, wt, share, a1, m1, a2, m2 in rows:
        print(f"{n:<14}{wt:>+9.3f}{share:>10.1f}%{a1:>10.1f}% /{m1:>6.3f}{a2:>10.1f}% /{m2:>6.3f}")
    print(f"\n(all four features: test MSE = {base_mse:.3f})")
    best = max(rows, key=lambda r: r[2])
    print(f"Most influential feature by weight: {best[0]}.  A larger rise in MSE when a feature is dropped "
          f"means the network relies on it more.")


# ----------------------------------------------------------------------------------------
# 4. Noise Filtering Challenge
# ----------------------------------------------------------------------------------------
def opt4():
    print("\n=== 4. Noise Filtering Challenge ===")
    print("Gaussian noise (in units of feature standard deviations) is added to the test inputs.")
    k = ask("Number of repeated readings averaged by the filter", 5, int)
    trials = ask("Trials per noise level", 30, int)
    m = get_main()
    yte = app.y[app.te]
    Zte, Ztr, ytr = app.Z[app.te], app.Z[app.tr], app.y[app.tr]
    rng = np.random.default_rng(7)
    print(f"\n{'noise sd':>9}{'A: clean-trained':>18}{'B: + averaging':>16}{'C: noise-trained':>18}   (test accuracy %)")
    for sigma in (0.0, 0.25, 0.5, 1.0, 1.5, 2.0):
        a, b, c = [], [], []
        mC = train_clf(Z=noisy(np.vstack([Ztr] * 5), sigma, rng), y=np.concatenate([ytr] * 5), epochs=60)
        for _ in range(trials):
            a.append(acc(yte, m.predict_class(noisy(Zte, sigma, rng))))
            avg = np.mean([noisy(Zte, sigma, rng) for _ in range(k)], axis=0)    # averaging filter
            b.append(acc(yte, m.predict_class(avg)))
            c.append(acc(yte, mC.predict_class(noisy(Zte, sigma, rng))))
        print(f"{sigma:>9.2f}{np.mean(a) * 100:>18.1f}{np.mean(b) * 100:>16.1f}{np.mean(c) * 100:>18.1f}")
    print("\nA = model trained on clean data, tested on noisy inputs.")
    print(f"B = same model, but each test input is the average of {k} noisy readings (noise sd shrinks by sqrt({k})).")
    print("C = ADALINE trained on noise-corrupted data (LMS learns to average the noise out).")


# ----------------------------------------------------------------------------------------
# 5. Misclassification Pattern Analyzer
# ----------------------------------------------------------------------------------------
def opt5():
    print("\n=== 5. Misclassification Pattern Analyzer ===")
    m = get_main()
    sigma = ask("Noise level for stress test (std units; 0 = clean data only)", 1.0)
    rng = np.random.default_rng(3)
    for label, Z in (("clean data", app.Z), (f"noisy data (sd={sigma})", noisy(app.Z, sigma, rng) if sigma > 0 else None)):
        if Z is None:
            continue
        out = m.net(Z)[:, 0]
        pred = (out >= 0.5).astype(int)
        bad = np.where(pred != app.y)[0]
        print(f"\n--- All 100 samples, {label}: {len(bad)} misclassified ---")
        if len(bad) == 0:
            print("   none - Setosa and Versicolor are linearly separable, so a clean ADALINE gets every sample right.")
            continue
        fp = int(((app.y == 0) & (pred == 1)).sum())
        fn = int(((app.y == 1) & (pred == 0)).sum())
        print(f"   Setosa -> predicted Versicolor: {fp}   Versicolor -> predicted Setosa: {fn}")
        print(f"   {'idx':>4} {'actual':>16} {'output':>8}  SL/SW/PL/PW (cm, original sample)")
        for i in bad[:12]:
            r = app.X[i]
            print(f"   {i:>4} {CLASS_NAME[app.y[i]]:>16} {out[i]:>8.3f}  {r[0]:.1f}/{r[1]:.1f}/{r[2]:.1f}/{r[3]:.1f}")
        if len(bad) > 12:
            print(f"   ... {len(bad) - 12} more")
        good = np.setdiff1d(np.arange(len(app.y)), bad)
        print("   Mean z-score, misclassified vs correct (which features push samples across the boundary):")
        for j, n in enumerate(FEATURES):
            print(f"      {n:<14}{Z[bad, j].mean():>+7.2f} vs {Z[good, j].mean():>+7.2f}")
        print(f"   Mean |output - 0.5|: misclassified {np.mean(np.abs(out[bad] - 0.5)):.3f}, "
              f"correct {np.mean(np.abs(out[good] - 0.5)):.3f}")


# ----------------------------------------------------------------------------------------
# 6. Decision Boundary Analyzer
# ----------------------------------------------------------------------------------------
def opt6():
    print("\n=== 6. Decision Boundary Analyzer ===")
    m = get_main()
    w, b = m.W[:, 0], m.b[0]
    w_o = w / app.sd
    b_o = b - np.sum(w * app.mu / app.sd)
    print("Boundary of the 4-feature model, in original units (cm):")
    print("   " + " ".join(f"{c:+.4f}*{n.replace(' ', '')}" for c, n in zip(w_o, FEATURES)) + f" {b_o:+.4f} = 0.5")
    dist = (m.net(app.Z)[:, 0] - 0.5) / np.linalg.norm(w)       # signed distance in standardised space
    side_ok = ((dist >= 0) == (app.y == 1))
    print(f"Samples on the correct side: {int(side_ok.sum())} / {len(app.y)}")
    print(f"Margin (distance to boundary, std units): Setosa mean {np.abs(dist[app.y == 0]).mean():.2f}, "
          f"Versicolor mean {np.abs(dist[app.y == 1]).mean():.2f}")
    print("\n5 samples closest to the boundary:")
    print(f"   {'idx':>4} {'actual':>16} {'distance':>9}")
    for i in np.argsort(np.abs(dist))[:5]:
        print(f"   {i:>4} {CLASS_NAME[app.y[i]]:>16} {dist[i]:>+9.3f}")

    # 2-D boundary on Petal Length (x) vs Petal Width (y)
    cols = (2, 3)
    m2 = train_clf(cols=cols, epochs=200)
    w2, b2 = m2.W[:, 0] / app.sd[list(cols)], m2.b[0] - np.sum(m2.W[:, 0] * app.mu[list(cols)] / app.sd[list(cols)])
    print("\n2-feature ADALINE (Petal Length, Petal Width):")
    print(f"   boundary: {w2[0]:+.4f}*PL {w2[1]:+.4f}*PW {b2:+.4f} = 0.5")
    if abs(w2[1]) > 1e-9:
        print(f"   i.e. PW = {-w2[0] / w2[1]:+.4f}*PL {(0.5 - b2) / w2[1]:+.4f}")
    xs, ys = app.X[:, cols[0]], app.X[:, cols[1]]
    W_, H_ = 60, 18
    x0, x1, y0, y1 = xs.min() - 0.2, xs.max() + 0.2, ys.min() - 0.1, ys.max() + 0.1
    grid = [[" "] * W_ for _ in range(H_)]
    if abs(w2[1]) > 1e-9:
        for c in range(W_):
            xv = x0 + (x1 - x0) * c / (W_ - 1)
            yv = (0.5 - b2 - w2[0] * xv) / w2[1]
            r = int(round((y1 - yv) / (y1 - y0) * (H_ - 1)))
            if 0 <= r < H_:
                grid[r][c] = "."
    for xv, yv, cl in zip(xs, ys, app.y):
        c = int(round((xv - x0) / (x1 - x0) * (W_ - 1)))
        r = int(round((y1 - yv) / (y1 - y0) * (H_ - 1)))
        grid[r][c] = "S" if cl == 0 else "V"
    print(f"\n   Petal Width ^   (S = Setosa, V = Versicolor, . = decision boundary)")
    for row in grid:
        print("   |" + "".join(row))
    print("   +" + "-" * W_ + "> Petal Length")


# ----------------------------------------------------------------------------------------
# 7. Comparative Analysis
# ----------------------------------------------------------------------------------------
def opt7():
    print("\n=== 7. Comparative Analysis ===")
    Ztr, Zte = app.Z[app.tr], app.Z[app.te]
    ytr, yte = app.y[app.tr], app.y[app.te]
    rows = []

    def add(name, model, cols, one_hot=False):
        Xte = Zte[:, list(cols)]
        if one_hot:
            p = np.argmax(model.net(Xte), axis=1)
            mse = np.mean((np.eye(2)[yte] - model.net(Xte)) ** 2)
        else:
            p = model.predict_class(Xte)
            mse = np.mean((yte - model.net(Xte)[:, 0]) ** 2)
        rows.append((name, acc(yte, p) * 100, float(mse), len(model.mse_history)))

    add("ADALINE, 1 input (Petal Length)", train_clf((2,), epochs=200), (2,))
    add("ADALINE, 2 inputs (PL, PW)", train_clf((2, 3), epochs=200), (2, 3))
    add("Multi-input ADALINE (4 inputs)", train_clf((0, 1, 2, 3), epochs=200), (0, 1, 2, 3))
    sl = Adaline(4, 2, 0.01, 200).fit(Ztr, np.eye(2)[ytr])
    add("Single-layer ADALINE (4 in, 2 out)", sl, (0, 1, 2, 3), one_hot=True)
    add("Multi-input, high lr (0.1)", train_clf((0, 1, 2, 3), lr=0.1, epochs=200), (0, 1, 2, 3))
    add("Multi-input, low lr (0.001)", train_clf((0, 1, 2, 3), lr=0.001, epochs=200), (0, 1, 2, 3))
    print(f"\n{'model':<38}{'test acc %':>11}{'test MSE':>10}{'epochs':>8}")
    for n, a, mse, ep in rows:
        print(f"{n:<38}{a:>11.1f}{mse:>10.4f}{ep:>8}")
    print("\nAccuracy is near-perfect for every model because Setosa/Versicolor are linearly separable; MSE and "
          "epochs-to-converge are what separate the models. More informative inputs lower MSE, and the learning "
          "rate trades convergence speed against stability.")


MENU = """
============ ADALINE Iris Classification System ============
 1. Binary Pattern Classification
 2. Multi-Input ADALINE Feature Prediction
 3. Feature Influence Analyzer
 4. Noise Filtering Challenge
 5. Misclassification Pattern Analyzer
 6. Decision Boundary Analyzer
 7. Comparative Analysis
 8. Exit
============================================================"""
ACTIONS = {"1": opt1, "2": opt2, "3": opt3, "4": opt4, "5": opt5, "6": opt6, "7": opt7}


def main():
    setup(sys.argv[1] if len(sys.argv) > 1 else None)
    while True:
        print(MENU)
        try:
            c = input("Enter your choice (1-8): ").strip()
        except EOFError:
            break
        if c == "8":
            print("Goodbye!")
            break
        fn = ACTIONS.get(c)
        if fn is None:
            print("Invalid choice, please enter a number from 1 to 8.")
            continue
        fn()


if __name__ == "__main__":
    main()

Loaded scikit-learn built-in Iris: 100 samples (50 Setosa + 50 Versicolor), 4 features. Train 70, test 30.

============ ADALINE Iris Classification System ============
 1. Binary Pattern Classification
 2. Multi-Input ADALINE Feature Prediction
 3. Feature Influence Analyzer
 4. Noise Filtering Challenge
 5. Misclassification Pattern Analyzer
 6. Decision Boundary Analyzer
 7. Comparative Analysis
 8. Exit
Enter your choice (1-8): 1

=== 1. Binary Pattern Classification (Setosa = 0, Versicolor = 1) ===
Learning rate [default 0.01]: 0.1
Max epochs [default 100]: 20

Epochs run: 20
   epoch   1: MSE = 0.01883
   epoch   5: MSE = 0.01677
   epoch  10: MSE = 0.01235
   epoch  20: MSE = 0.01383

Learned weights (standardised inputs):
   Sepal Length  -0.0376
   Sepal Width   -0.1131
   Petal Length  +0.2426
   Petal Width   +0.1745
   Bias          +0.5405

Training accuracy: 100.00%   Test accuracy: 100.00%
Confusion matrix:        Pred Setosa  Pred Versicolor
   Actual Setosa            